In [1]:
# Importar librerías
import pandas as pd
from src.config import data_folder
from src.funcionesExtract import *

In [2]:
# Obtener lista de tickers del S&P 500 desde el fichero constituents.csv
# Si no existe el fichero, lo descarga desde GitHub
# Para actualizar la lista de componentes, cambiar force_update a True, ejecutar y volver a dejar en False
ruta_archivo = descargar_constituents(force_update=False) 

# Cargar datos
df_tickers = pd.read_csv(ruta_archivo)
df_tickers.info()

Usando archivo constituents.csv local.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Symbol                 503 non-null    object
 1   Security               503 non-null    object
 2   GICS Sector            503 non-null    object
 3   GICS Sub-Industry      503 non-null    object
 4   Headquarters Location  503 non-null    object
 5   Date added             503 non-null    object
 6   CIK                    503 non-null    int64 
 7   Founded                503 non-null    object
dtypes: int64(1), object(7)
memory usage: 31.6+ KB


In [3]:
# Limpieza previa de df_tickers
df_tickers_clean = limpieza_tickers(df_tickers)

# Lista de tickers
tickers_list = df_tickers_clean["Ticker"].tolist()
df_tickers_clean.head()

,Ticker,Sector,DateAdded
0,MMM,Industrials,1957-03-04
1,AOS,Industrials,2017-07-26
2,ABT,HealthCare,1957-03-04
3,ABBV,HealthCare,2012-12-31
4,ACN,InformationTechnology,2011-07-06


In [4]:
# Extraer precios de los tickers y del índice SPY (demora unos 3 minutos)
df_prices = extraer_precios(tickers_list)

# Se extrae precio del Índice de Mercado para usar en cálculos y se guarda en fichero
df_index = extraer_precios(['SPY'])
df_index.to_parquet(f"{data_folder}/market_index.parquet")

df_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35755 entries, 0 to 35754
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       35755 non-null  datetime64[ns]
 1   Close      35755 non-null  float64       
 2   Dividends  35755 non-null  float64       
 3   Ticker     35755 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(1)
memory usage: 1.1+ MB


In [5]:
df_index.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       72 non-null     datetime64[ns]
 1   Close      72 non-null     float64       
 2   Dividends  72 non-null     float64       
 3   Ticker     72 non-null     object        
dtypes: datetime64[ns](1), float64(2), object(1)
memory usage: 2.4+ KB


In [6]:
# Unir df_prices y df_tickers
df_merged = pd.merge(
    df_prices,
    df_tickers_clean,
    on= 'Ticker',
    how= 'left'
)
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35755 entries, 0 to 35754
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       35755 non-null  datetime64[ns]
 1   Close      35755 non-null  float64       
 2   Dividends  35755 non-null  float64       
 3   Ticker     35755 non-null  object        
 4   Sector     35755 non-null  object        
 5   DateAdded  35755 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(3)
memory usage: 1.6+ MB


In [7]:
# Extraer datos financieros de yfinance (ultimos 4 reportes trimestrales, demora unos 10 minutos)
df_yfinance = extraer_financials(tickers_list)
df_yfinance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2010 entries, 0 to 2009
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Date                       2010 non-null   datetime64[ns]
 1   Total Revenue              2001 non-null   float64       
 2   Gross Profit               1795 non-null   float64       
 3   Operating Income           1818 non-null   float64       
 4   Net Income                 2004 non-null   float64       
 5   EBITDA                     1818 non-null   float64       
 6   Basic Average Shares       1963 non-null   float64       
 7   Cash And Cash Equivalents  2001 non-null   float64       
 8   Current Debt               1523 non-null   float64       
 9   Long Term Debt             1877 non-null   float64       
 10  Total Debt                 1972 non-null   float64       
 11  Stockholders Equity        1998 non-null   float64       
 12  Total 

In [9]:
# Extraer datos de SimFin (trimestrales de los últimos 4 años con un lag de 1 año)
df_simfin_raw = extraer_simfin(tickers_list)
df_simfin_raw.info()

Dataset "us-income-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-balance-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-quarterly" on disk (0 days old).
- Loading from disk ... Done!
<class 'pandas.core.frame.DataFrame'>
Index: 6288 entries, 0 to 47602
Data columns (total 65 columns):
 #   Column                                           Non-Null Count  Dtype         
---  ------                                           --------------  -----         
 0   Ticker                                           6288 non-null   object        
 1   Report Date                                      6288 non-null   datetime64[ns]
 2   SimFinId                                         6267 non-null   float64       
 3   Currency                                         6267 non-null   object        
 4   Fiscal Year                                      6267 non-null   float64       
 5   Fiscal Period                                   

In [10]:
# Definir columnas a mantener en simfin para que coincidan
cols_a_mantener = df_yfinance.columns
df_simfin_clean = estandarizar_simfin(df_simfin_raw, cols_a_mantener)

# Eliminar posibles duplicados INTERNOS en cada DataFrame antes de cruzarlos
# Se mantiene el último como válido (dato reexpresado/corregido)
df_sf_unique = df_simfin_clean.drop_duplicates(subset=['Ticker', 'Date'], keep='last') 
df_yf_unique = df_yfinance.drop_duplicates(subset=['Ticker', 'Date'], keep='last')

# Convertir 'Ticker' y 'Date' en el índice de ambos DataFrames temporalmente
df_sf_idx = df_sf_unique.set_index(['Ticker', 'Date'])
df_yf_idx = df_yf_unique.set_index(['Ticker', 'Date'])


# Aplicar combine_first para tratar el solapamiento
# Toma el valor de yfinance, si fuese NaN lo intenta rellenar con SimFin
df_financials_idx = df_yf_idx.combine_first(df_sf_idx)

# Se devuelven 'Ticker' y 'Date' como columnas normales
df_financials_completo = df_financials_idx.reset_index()
# Auditoría de solapamientos
mascara_duplicados = df_financials_completo.duplicated(subset=['Ticker', 'Date'], keep=False)
df_solapado = df_financials_completo[mascara_duplicados]
print(f"Se han encontrado {len(df_solapado)} filas con Ticker y Date solapados.")

Se han encontrado 0 filas con Ticker y Date solapados.


In [11]:
# Unir dataframe de precios con datos financieros
df_final = pd.merge(
    df_merged, 
    df_financials_completo, 
    on=['Date', 'Ticker'],
    how='left'
)
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35755 entries, 0 to 35754
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Date                       35755 non-null  datetime64[ns]
 1   Close                      35755 non-null  float64       
 2   Dividends                  35755 non-null  float64       
 3   Ticker                     35755 non-null  object        
 4   Sector                     35755 non-null  object        
 5   DateAdded                  35755 non-null  object        
 6   Total Revenue              8253 non-null   float64       
 7   Gross Profit               7748 non-null   float64       
 8   Operating Income           8070 non-null   float64       
 9   Net Income                 8256 non-null   float64       
 10  EBITDA                     4882 non-null   float64       
 11  Basic Average Shares       8209 non-null   float64       
 12  Cash

In [12]:
# Limpieza final, con forward fill para datos financieros
df_final_clean = limpieza_final(df_final)
df_final_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14626 entries, 0 to 14625
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Date                    14626 non-null  datetime64[ns]
 1   Close                   14626 non-null  float64       
 2   Dividends               14626 non-null  float64       
 3   Ticker                  14626 non-null  object        
 4   Sector                  14626 non-null  object        
 5   DateAdded               14626 non-null  object        
 6   TotalRevenue            14617 non-null  float64       
 7   GrossProfit             14105 non-null  float64       
 8   OperatingIncome         14617 non-null  float64       
 9   NetIncome               14626 non-null  float64       
 10  EBITDA                  14626 non-null  float64       
 11  BasicAverageShares      14618 non-null  float64       
 12  CashAndCashEquivalents  14614 non-null  float6

In [13]:
# Guardar datos extraidos en fichero raw_data
df_final_clean.to_parquet(f"{data_folder}/raw_data.parquet")
print("Fichero guardado en la carpeta",data_folder)

Fichero guardado en la carpeta data
